# D167 — Understanding Cumulative Distribution with `CUME_DIST`

A **distribution** describes how values are spread. **Cumulative** means building up through the current value.

`CUME_DIST` returns the proportion of rows that have reached the current ordered value. Its result is greater than 0 and no more than 1. Multiplying by 100 gives a cumulative percentage.

With ascending order:

```text
CUME_DIST = rows with value less than or equal to current value / all rows
```

With descending order, it becomes the share of rows with a value greater than or equal to the current value. Tied values receive the same cumulative distribution, measured at the end of the tie group.

## When is cumulative distribution useful?

It answers questions such as:

- What percentage of orders cost no more than this amount?
- What share of requests finished within this response time?
- What proportion of students scored at or below this score?
- Which price is enough to cover at least 90% of observations?
- What share of sellers have revenue greater than or equal to this seller?

This is useful for thresholds, service-level targets, distribution charts, and identifying percentiles.

In [ ]:
import os
import mysql.connector
connection=mysql.connector.connect(
 host=os.environ.get('MYSQL_HOSTNAME','127.0.0.1'),
 port=int(os.environ.get('MYSQL_PORT','3306')),
 user=os.environ.get('MYSQL_USERNAME','root'),
 password=os.environ.get('MYSQL_PASSWORD','root'),
 database=os.environ.get('MYSQL_DATABASE','olist_import_lab'))
print('Connected:',connection.is_connected())

In [ ]:
def execute_sql(sql):
    cursor=connection.cursor(); cursor.execute(sql)
    columns=[x[0] for x in cursor.description]; rows=cursor.fetchall(); cursor.close()
    text=[[str(v) for v in row] for row in rows]; widths=[len(c) for c in columns]
    for row in text: widths=[max(w,len(v)) for w,v in zip(widths,row)]
    print(' | '.join(c.ljust(w) for c,w in zip(columns,widths)))
    print('-+-'.join('-'*w for w in widths))
    for row in text: print(' | '.join(v.ljust(w) for v,w in zip(row,widths)))
    return rows

## Example 1 — Scores, ties, and the manual calculation

For ascending scores, cumulative distribution is the share scoring at or below the current score. There are five rows. The two scores of 80 are tied, so both are measured at the end of that tie: 3 out of 5 rows, or 60%.

In [ ]:
execute_sql("""
WITH scores AS (
 SELECT 'Asha' student,60 score UNION ALL SELECT 'Bilal',80 UNION ALL
 SELECT 'Chen',80 UNION ALL SELECT 'Divya',90 UNION ALL SELECT 'Eshan',100
)
SELECT student,score,
 ROUND(CUME_DIST() OVER(ORDER BY score ASC)*100,1) cumulative_percent
FROM scores ORDER BY score,student
""")

## Example 2 — Service response target

A service team wants to know the share of requests completed within each response time. Ascending order matches the phrase “less than or equal to this time.” At 500 milliseconds, the cumulative percentage shows how much traffic meets a 500 ms target.

In [ ]:
execute_sql("""
WITH responses AS (
 SELECT 'A' request_name,100 milliseconds UNION ALL SELECT 'B',180 UNION ALL
 SELECT 'C',250 UNION ALL SELECT 'D',500 UNION ALL SELECT 'E',500 UNION ALL
 SELECT 'F',900 UNION ALL SELECT 'G',1500
)
SELECT request_name,milliseconds,
 ROUND(CUME_DIST() OVER(ORDER BY milliseconds)*100,1) completed_within_percent
FROM responses ORDER BY milliseconds,request_name
""")

## Example 3 — Find the first value reaching 80%

A CTE calculates cumulative distribution. The outer query keeps values reaching at least 80%, and `LIMIT 1` selects the first such threshold. This is one way to locate an approximate percentile from observed values.

In [ ]:
execute_sql("""
WITH delivery_days AS (
 SELECT 1 days UNION ALL SELECT 2 UNION ALL SELECT 2 UNION ALL SELECT 3 UNION ALL
 SELECT 4 UNION ALL SELECT 5 UNION ALL SELECT 5 UNION ALL SELECT 7 UNION ALL
 SELECT 9 UNION ALL SELECT 12
), distributed AS (
 SELECT days,CUME_DIST() OVER(ORDER BY days) cd FROM delivery_days
)
SELECT days AS first_days_reaching_80_percent,ROUND(cd*100,1) cumulative_percent
FROM distributed WHERE cd>=0.80
ORDER BY days LIMIT 1
""")

## Example 4 — Separate distributions by category

`PARTITION BY category` starts a fresh distribution inside each category. A price of 50 may have a different position among books than among electronics.

In [ ]:
execute_sql("""
WITH prices AS (
 SELECT 'Books' category,'B1' item,10 price UNION ALL SELECT 'Books','B2',20 UNION ALL
 SELECT 'Books','B3',50 UNION ALL SELECT 'Electronics','E1',50 UNION ALL
 SELECT 'Electronics','E2',200 UNION ALL SELECT 'Electronics','E3',500 UNION ALL
 SELECT 'Electronics','E4',1000
)
SELECT category,item,price,ROUND(CUME_DIST() OVER(
 PARTITION BY category ORDER BY price)*100,1) category_cumulative_percent
FROM prices ORDER BY category,price
""")

## Olist example — Item price distribution

For ascending item prices, cumulative percentage answers: “What share of Olist item rows have a price less than or equal to this price?” The query groups identical prices first so the output is compact, then calculates the cumulative row share using a running sum.

This grouped form gives the same distribution meaning as `CUME_DIST` on every item row, while showing one row per distinct price.

In [ ]:
execute_sql("""
WITH price_counts AS (
 SELECT price,COUNT(*) item_count
 FROM olist_order_items GROUP BY price
), distributed AS (
 SELECT price,item_count,
  SUM(item_count) OVER(ORDER BY price) cumulative_items,
  SUM(item_count) OVER() all_items
 FROM price_counts
)
SELECT price,item_count,cumulative_items,
 ROUND(100.0*cumulative_items/all_items,2) cumulative_percent
FROM distributed
WHERE cumulative_items/all_items>=0.90
ORDER BY price LIMIT 10
""")

## Olist example — Seller distribution with `CUME_DIST`

Descending seller value changes the interpretation: the result is the share of sellers having item value greater than or equal to the current seller. The highest seller therefore has a small positive percentage, not zero.

In [ ]:
execute_sql("""
WITH seller_totals AS (
 SELECT seller_id,SUM(price) item_value
 FROM olist_order_items GROUP BY seller_id
), distributed AS (
 SELECT seller_id,item_value,CUME_DIST() OVER(ORDER BY item_value DESC) cd
 FROM seller_totals
)
SELECT seller_id,ROUND(item_value,2) item_value,
 ROUND(cd*100,3) sellers_at_or_above_percent
FROM distributed WHERE cd<=0.01
ORDER BY item_value DESC
""")

## `PERCENT_RANK` compared with `CUME_DIST`

| Question | Function |
|---|---|
| Where does this rank sit between the first and last rank? | `PERCENT_RANK` |
| What share of rows has reached this ordered value? | `CUME_DIST` |
| Can the first result be 0? | `PERCENT_RANK`: yes |
| Is the final cumulative result 1? | `CUME_DIST`: yes |
| How are ties handled? | both give tied values the same result, but use different formulas |

Use `CUME_DIST` for threshold and coverage questions. Use `PERCENT_RANK` for relative ranked position.

In [ ]:
connection.close()
print('MySQL connection closed.')